本章回答三个问题：

1. LangChain 和 LangGraph 是什么关系？该用哪个？（本章第 1 节）
2. 一个 Graph 由哪些基本要素构成？（本章第 2 节）
3. Graph 是怎么运行起来的？（本章第 3 节）

后续安排：第 2 章深入 State（reducer、多 schema），第 3 章讲控制流（分支、循环）。本章的「Graph 要素 + 运行过程」是后两章的地基，读完本章你就能写出一个最小的可运行 Graph。

# 1 LangChain 与 LangGraph

## 1.1 它们是什么

- **LangChain**：构建 LLM 应用的组件库（model 调用、prompt template、output parser、document 处理、vector store 等）
- **LangGraph**：LangChain 生态里的 Graph 编排框架，以「Graph + State」为核心，解决了早期 Chain 的局限

## 1.2 为什么需要 LangGraph

LangChain 早期的 **Chain** 是线性管道 A → B → C，真实场景很快不够用：

- 没有**循环**：ReAct 这类 agent 需要「思考 → 行动 → 观察 → 再思考」
- 没有**分支**：不同输入走不同路径
- 没有**State**：步骤之间无法保存和更新上下文，更无法持久化
- 没有**人工介入**：无法在关键 Node 停下来等用户确认

LangGraph 用 Graph 取代 Chain：Node 是步骤，Edge 是路径，State 在整个 Graph 上流动，循环、分支、人工介入都直接支持。

## 1.3 定位：不是替代，是分工

| 维度 | LangChain | LangGraph |
|---|---|---|
| 定位 | 通用组件库 | 流程编排框架 |
| 核心抽象 | Chain / Runnable | StateGraph（directed graph） |
| 执行模型 | 线性管道 | Graph 遍历，支持循环/分支/并发 |
| State 管理 | 无（数据在 Chain 内传递） | 显式 State，可增量更新、可持久化 |
| 典型场景 | 简单问答、RAG pipeline | Agent、多 agent、人工审批流程 |
| 关系 | 提供组件 | 使用 LangChain 组件编排 |

**结论**：

- 固定线性步骤 → LangChain Chain 就够
- 需要循环、分支、State、人工介入 → LangGraph
- 复杂 agent 的主流选择是 LangGraph

# 2 Graph 的基本要素

一个 Graph 由 4 个要素构成：

| 要素 | 含义 | 代码对应 |
|---|---|---|
| **State** | 所有 Node 共享的数据容器 | 一个 TypedDict 或 Pydantic 类 |
| **Node** | 一个工作单元：普通 Python 函数 | `graph.add_node(...)` |
| **Edge** | Node 间的连接路径，决定执行顺序 | `graph.add_edge(...)` / `add_conditional_edges(...)` |
| **START / END** | 特殊 Node：Graph 的入口和出口 | `START`、`END` 常量 |

## 2.1 State

State 是一个类型定义，描述 Graph 上流动的数据。每个 Node 读 State、返回**部分更新**，LangGraph 自动把更新合并回 State：

```python
from typing_extensions import TypedDict

class ChatState(TypedDict):
    messages: list[str]   # 对话历史
    count: int            # 处理轮数
```

## 2.2 Node

Node 就是一个函数：**接收 State，返回要更新的字段**：

```python
def node_a(state: ChatState) -> dict:
    return {"count": state["count"] + 1}   # 只返回要更新的字段
```

## 2.3 Edge

- **普通 Edge**：`A → B`，A 跑完一定跑 B
- **条件 Edge**：A 跑完后根据返回值**选择**下一个 Node

## 2.4 拼装 StateGraph

```python
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)          # 1. 创建 Graph，声明 State 类型
builder.add_node("node_a", node_a)      # 2. 添加 Node
builder.add_edge(START, "node_a")       # 3. 连接：入口 → Node
builder.add_edge("node_a", END)         # 4. 连接：Node → 出口
app = builder.compile()                  # 5. 编译
```

下面用完整代码演示一个「两个 Node + State 累积」的最小 Graph：

In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class OverAllState(TypedDict):
    logs: list[str]
    cur_id: str


def node_1(state: OverAllState) -> dict:
    # 无 reducer 的 key 是「后写覆盖先写」，想保留旧值就自己读出来拼上
    return {
        "cur_id": state["cur_id"] + ", node_1",
        "logs": state["logs"] + ["node_1 运行完毕"],
    }


def node_2(state: OverAllState) -> dict:
    return {
        "cur_id": state["cur_id"] + ", node_2",
        "logs": state["logs"] + ["node_2 运行完毕"],
    }


builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

result = builder.compile().invoke({"cur_id": "start", "logs": []})
print(result)
# {'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}

# 「自己读旧值拼新值」很啰嗦。第 2 章的 reducer（Annotated[T, add]）可以让 Node 只返回增量，
# 合并交给 LangGraph 自动完成。

# 3 Graph 运行过程

上面的代码只做了两件事：`compile()` 检查图结构是否合法，`invoke(初始 State)` 才是真正的运行。一次 invoke 的过程：

1. **初始化**：invoke 的入参先写入 State
2. **调度**：从 START 出发，把下一个可执行的 Node 放入执行队列
3. **执行与合并**：按边依次执行 Node，每个 Node 返回的更新**立即合并**进 State
4. **结束**：到达 END 后停止，返回完整的最终 State

于是 `invoke({"cur_id": "start", "logs": []})` 会依次执行 node_1 → node_2：node_1 的返回值先合并进 State，node_2 读到的就是合并后的 State，以此类推。

> 上面的 Graph 是线性执行。循环、分支、并行会让「调度」变得复杂（super-step 等细节），第 3 章会展开讲。